# Data Acquisition, Merge, and Checks for NYISO and ERA5 Data

## Imports and Configuration

In [1]:
import os
import zipfile
from pathlib import Path

import pandas as pd

In [2]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_NYISOSolarForecast"

data_root = repo_root / "data"
raw_dir = data_root / "raw"
processed_dir = data_root / "processed"

solar_raw_dir = raw_dir / "nyiso_solar"

In [3]:
nyiso_out = processed_dir / "01_nyiso_merged.csv"
era5_src = processed_dir / "02_era5_weather.csv"
era5_out = processed_dir / "02_era5_weather.csv"
merged_out = processed_dir / "03_merged_data.csv"

processed_dir.mkdir(parents=True, exist_ok=True)

## Data Extraction 

In [4]:
solar_zip_path = raw_dir / "nyiso_solar.zip"

unzipped_dirs = {
    "actuals": solar_raw_dir / "unzipped_actuals",
    "forecasts": solar_raw_dir / "unzipped_forecasts",
    "capacity": solar_raw_dir / "unzipped_capacity",
}

### Converting ZIP to CSV

In [5]:
if solar_zip_path.exists():
    solar_raw_dir.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(solar_zip_path, "r") as archive:
            archive.extractall(solar_raw_dir)
        print(f"Extracted Main: {solar_zip_path}")
    except Exception as e:
        print(f"Didn't Extract Main: {solar_zip_path.name} | {e}")
else:
    print(f"Not Found: {solar_zip_path}")

Extracted Main: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/raw/nyiso_solar.zip


In [6]:
def unzip_all_archives(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    extracted_files = 0

    if not os.path.exists(input_folder):
        print(f"Input folder not found: {input_folder}")
        return

    for filename in os.listdir(input_folder):
        if filename.endswith(".zip"):
            try:
                with zipfile.ZipFile(os.path.join(input_folder, filename), "r") as archive:
                    archive.extractall(output_folder)
                    extracted_files += 1
            except Exception as e:
                print(f"Did Not Extract: {filename} | {e}")

    print(f"Extraction Completed: {extracted_files} archives from {input_folder}")

### Load CSV Files

In [25]:
def load_folder(folder: Path) -> pd.DataFrame:
    csv_files = list(folder.glob("*.csv"))
    frames = []

    for file in csv_files:
        try:
            df = pd.read_csv(file)
            df["source_file"] = file.name
            frames.append(df)
        except Exception as e:
            print(f"Failed to Read: {file.name} | {e}")

    if not frames:
        return pd.DataFrame()

    df_all = pd.concat(frames, ignore_index=True)
    return df_all

In [26]:
df_actual = load_folder(unzipped_dirs["actuals"])
df_forecast = load_folder(unzipped_dirs["forecasts"])
df_capacity = load_folder(unzipped_dirs["capacity"])

In [27]:
print("Loaded Shapes")
print(". . .")
print("Actuals:", df_actual.shape)
print("Forecasts:", df_forecast.shape)
print("Capacity:", df_capacity.shape)

Loaded Shapes
. . .
Actuals: (500916, 5)
Forecasts: (506004, 5)
Capacity: (10716, 5)


## Data Standardization

In [28]:
ts_col = "time_stamp"
zone_col = "zone_name"
val_col = "mw_value"

### Column Name Treatment

In [29]:
for df in (df_actual, df_forecast, df_capacity):
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

def ensure_required_columns(df: pd.DataFrame, df_name: str) -> None:
    missing = [col for col in [ts_col, zone_col] if col not in df.columns]
    if missing:
        raise KeyError(f"{df_name} is missing required columns: {missing}. Found: {df.columns.tolist()}")

ensure_required_columns(df_actual, "df_actual")
ensure_required_columns(df_forecast, "df_forecast")
ensure_required_columns(df_capacity, "df_capacity")

In [30]:
def resolve_value_col(df: pd.DataFrame) -> str:
    candidates = [
        "mw_value",
        "mw",
        "value",
        "actual_mw",
        "forecast_mw",
        "capacity_mw",
        "name",
    ]

    for col in candidates:
        if col in df.columns:
            return col

    numeric_candidates = []
    for col in df.columns:
        if col in [ts_col, zone_col, "source_file"]:
            continue
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().sum() > 0:
            numeric_candidates.append(col)

    if numeric_candidates:
        return numeric_candidates[0]

    raise KeyError(f"No megawatts columns, the columns were: {df.columns.tolist()}")

actual_val_col = resolve_value_col(df_actual)
forecast_val_col = resolve_value_col(df_forecast)
capacity_val_col = resolve_value_col(df_capacity)

### Timestamp Treatment

In [31]:
def parse_nyiso_time(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    raw_ts = pd.to_datetime(df[ts_col], errors="coerce")
    parsed_ts = pd.Series(pd.NaT, index=df.index, dtype="object")

    if "time_zone" in df.columns:
        tz_series = df["time_zone"].astype(str).str.upper().str.strip()

        is_est = tz_series.eq("EST")
        is_edt = tz_series.eq("EDT")
        other_mask = ~(is_est | is_edt)

        if is_est.any():
            parsed_ts.loc[is_est] = (
                pd.to_datetime(df.loc[is_est, ts_col], errors="coerce")
                .dt.tz_localize("Etc/GMT+5", nonexistent="shift_forward", ambiguous="NaT")
                .dt.tz_convert("UTC")
            )

        if is_edt.any():
            parsed_ts.loc[is_edt] = (
                pd.to_datetime(df.loc[is_edt, ts_col], errors="coerce")
                .dt.tz_localize("Etc/GMT+4", nonexistent="shift_forward", ambiguous="NaT")
                .dt.tz_convert("UTC")
            )

        if other_mask.any():
            parsed_ts.loc[other_mask] = (
                pd.to_datetime(df.loc[other_mask, ts_col], errors="coerce")
                .dt.tz_localize(
                    "America/New_York",
                    nonexistent="shift_forward",
                    ambiguous="NaT",
                )
                .dt.tz_convert("UTC")
            )
    else:
        parsed_ts = (
            raw_ts
            .dt.tz_localize(
                "America/New_York",
                nonexistent="shift_forward",
                ambiguous="NaT",
            )
            .dt.tz_convert("UTC")
            .astype("object")
        )

    df[ts_col] = pd.to_datetime(parsed_ts, utc=True, errors="coerce").dt.floor("h")

    df[zone_col] = (
        df[zone_col]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    return df


In [32]:
df_actual = parse_nyiso_time(df_actual)
df_forecast = parse_nyiso_time(df_forecast)
df_capacity = parse_nyiso_time(df_capacity)

df_actual[actual_val_col] = pd.to_numeric(df_actual[actual_val_col], errors="coerce")
df_forecast[forecast_val_col] = pd.to_numeric(df_forecast[forecast_val_col], errors="coerce")
df_capacity[capacity_val_col] = pd.to_numeric(df_capacity[capacity_val_col], errors="coerce")

df_actual = df_actual.rename(columns={actual_val_col: "actual_mw"})
df_forecast = df_forecast.rename(columns={forecast_val_col: "forecast_mw"})
df_capacity = df_capacity.rename(columns={capacity_val_col: "capacity_mw"})

In [33]:
print("\nColumns After Canonical Renaming")
print(". . .")
print("Actual:", df_actual.columns.tolist())
print("Forecast:", df_forecast.columns.tolist())
print("Capacity:", df_capacity.columns.tolist())


Columns After Canonical Renaming
. . .
Actual: ['time_stamp', 'time_zone', 'zone_name', 'actual_mw', 'source_file']
Forecast: ['time_stamp', 'time_zone', 'zone_name', 'forecast_mw', 'source_file']
Capacity: ['time_stamp', 'time_zone', 'zone_name', 'capacity_mw', 'source_file']


## Concatenating and Aggregating Data

### Merge All NYISO Data

In [41]:
df_actual_hourly = (
    df_actual
    .dropna(subset=[ts_col, zone_col, "actual_mw"])
    .groupby([ts_col, zone_col], as_index=False)["actual_mw"]
    .sum()
)

df_forecast_hourly = (
    df_forecast
    .dropna(subset=[ts_col, zone_col, "forecast_mw"])
    .groupby([ts_col, zone_col], as_index=False)["forecast_mw"]
    .sum()
)

df_capacity_updates = (
    df_capacity
    .dropna(subset=[ts_col, zone_col, "capacity_mw"])
    .sort_values([zone_col, ts_col, "source_file"])
    .groupby([ts_col, zone_col], as_index=False)["capacity_mw"]
    .last()
)

print("\nPost-Aggregation Shapes")
print(". . .")
print("Actual hourly:", df_actual_hourly.shape)
print("Forecast hourly:", df_forecast_hourly.shape)
print("Capacity updated:", df_capacity_updates.shape)


Post-Aggregation Shapes
. . .
Actual hourly: (500916, 3)
Forecast hourly: (506004, 3)
Capacity updated: (10716, 3)


In [42]:
df_nyiso = (
    df_actual_hourly
    .merge(df_forecast_hourly, how="outer", on=[ts_col, zone_col])
    .sort_values([zone_col, ts_col])
    .reset_index(drop=True)
)

df_nyiso = (
    df_nyiso
    .merge(df_capacity_updates, how="left", on=[ts_col, zone_col])
    .sort_values([zone_col, ts_col])
    .reset_index(drop=True)
)

df_nyiso["capacity_mw"] = (
    df_nyiso
    .groupby(zone_col)["capacity_mw"]
    .ffill()
)

df_nyiso.to_csv(nyiso_out, index=False)

### Add ERA5 Data

In [36]:
df_era5 = pd.read_csv(era5_src, low_memory=False)

df_era5.columns = (
    df_era5.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

if "time_stamp" in df_era5.columns:
    df_era5[ts_col] = pd.to_datetime(df_era5["time_stamp"], utc=True, errors="coerce")
elif "time" in df_era5.columns:
    df_era5[ts_col] = pd.to_datetime(df_era5["time"], utc=True, errors="coerce")
else:
    raise KeyError(f"ERA5 time not found. The columns were: {df_era5.columns.tolist()}")

df_era5[ts_col] = df_era5[ts_col].dt.floor("h")
df_era5 = df_era5.dropna(subset=[ts_col])
df_era5 = df_era5.groupby(ts_col, as_index=False).first()

df_era5.to_csv(era5_out, index=False)

## Final Dataset Validation

In [37]:
df_merge = pd.merge(df_nyiso, df_era5, on=ts_col, how="inner")
df_merge = df_merge.sort_values([ts_col, zone_col]).reset_index(drop=True)

df_merge.to_csv(merged_out, index=False)

print("NYISO Shape:", df_nyiso.shape)
print("ERA5 Shape:", df_era5.shape)
print("Merged Shape:", df_merge.shape)

print("Merged Time Range:", df_merge[ts_col].min(), "→", df_merge[ts_col].max())
print("Saved:", merged_out)

NYISO Shape: (509460, 5)
ERA5 Shape: (51000, 8)
Merged Shape: (509460, 12)
Merged Time Range: 2020-11-17 05:00:00+00:00 → 2025-09-21 03:00:00+00:00
Saved: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/processed/03_merged_data.csv


In [38]:
print("Columns Check")
print(". . .")
print("NYISO Columns:", df_nyiso.columns.tolist())
print("ERA5 Columns:", df_era5.columns.tolist())
print("Merged Columns:", df_merge.columns.tolist())

Columns Check
. . .
NYISO Columns: ['time_stamp', 'zone_name', 'actual_mw', 'forecast_mw', 'capacity_mw']
ERA5 Columns: ['time_stamp', 'time', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'year']
Merged Columns: ['time_stamp', 'zone_name', 'actual_mw', 'forecast_mw', 'capacity_mw', 'time', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'year']


In [39]:
print("Missing Check")
print(". . .")
print("NYISO Missing time_stamp:", df_nyiso[ts_col].isna().sum())
print("ERA5 Missing time_stamp:", df_era5[ts_col].isna().sum())
print("Merged Missing time_stamp:", df_merge[ts_col].isna().sum())

Missing Check
. . .
NYISO Missing time_stamp: 0
ERA5 Missing time_stamp: 0
Merged Missing time_stamp: 0


In [40]:
print("Saved NYISO:", nyiso_out)
print("Saved ERA5:", era5_out)
print("Saved merged:", merged_out)

Saved NYISO: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/processed/01_nyiso_merged.csv
Saved ERA5: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/processed/02_era5_weather.csv
Saved merged: /Users/Sumaitat/Documents/Coding/ML_NYISOSolarForecast/data/processed/03_merged_data.csv
